In [3]:
from transformers import pipeline

# --------------------------------------------------
# 1. Nạp BERT đã huấn luyện trước
# --------------------------------------------------

bert_fill_mask = pipeline(
    task="fill-mask",
    model="google-bert/bert-base-uncased"
)

# --------------------------------------------------
# 2. Chuẩn bị những câu có một từ bị che
# --------------------------------------------------

sentences = [
    "The fisherman sat on the [MASK] of the river.",
    "She deposited money at the [MASK].",
    "The sky is [MASK] today."
]

# --------------------------------------------------
# 3. Yêu cầu BERT dự đoán từ bị che trong mỗi câu
# --------------------------------------------------

for sentence in sentences:
    print("\nCâu đầu vào:", sentence)

    predictions = bert_fill_mask(
        sentence,
        top_k=5
    )

    print("Các dự đoán hàng đầu:")

    for item in predictions:
        print(
            "Từ dự đoán:", item["token_str"],
            "| Điểm:", round(item["score"], 4),
            "| Câu hoàn chỉnh:", item["sequence"]
        )

KeyboardInterrupt: 

In [4]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
 
MODEL = "bert-base-uncased"
 
# ---- Phần 1: điền từ bị che (Masked Language Modeling) ----
print("=== 1. Fill-Mask ===")
fill = pipeline("fill-mask", model=MODEL)
for r in fill("The capital of France is [MASK].", top_k=3):
    print(f"{r['token_str']:<10} score={r['score']:.4f}")
 
# ---- Phần 2: embedding theo ngữ cảnh (so sánh với ELMo) ----
print("\n=== 2. Contextual embedding ===")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL).eval()
 
def word_vector(sentence, word):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state[0]  # (seq_len, 768)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return hidden[tokens.index(word)]
 
v1 = word_vector("I deposited money at the bank", "bank")
v2 = word_vector("We sat on the bank of the river", "bank")
v3 = word_vector("The bank approved my loan", "bank")
 
cos = torch.nn.functional.cosine_similarity
print("bank (ngân hàng) vs bank (bờ sông)   :", cos(v1, v2, dim=0).item())
print("bank (ngân hàng) vs bank (ngân hàng) :", cos(v1, v3, dim=0).item())

=== 1. Fill-Mask ===


d:\NLP\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dai\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 874.03it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key    

paris      score=0.4168
lille      score=0.0714
lyon       score=0.0634

=== 2. Contextual embedding ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 674.36it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bank (ngân hàng) vs bank (bờ sông)   : 0.2999270558357239
bank (ngân hàng) vs bank (ngân hàng) : 0.5445905923843384


In [5]:
"""
Demo họ GPT (Generative Pre-trained Transformer).
Dùng GPT-2 (mã nguồn mở, chạy được trên máy cá nhân) để sinh văn bản.
GPT-3/4... không công khai trọng số nên chỉ gọi qua API (xem phần cuối file).

Cài đặt:
    pip install transformers torch

Chạy:
    python 03_gpt_demo.py
"""
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed

set_seed(42)
MODEL = "gpt2"  # có thể thử: gpt2-medium, gpt2-large

tokenizer = GPT2Tokenizer.from_pretrained(MODEL)
model = GPT2LMHeadModel.from_pretrained(MODEL).eval()

prompt = "Artificial intelligence will change the world because"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,      # lấy mẫu ngẫu nhiên thay vì greedy
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

# ---- (Tuỳ chọn) Gọi GPT qua API của OpenAI ----
# pip install openai ; đặt biến môi trường OPENAI_API_KEY
#
# from openai import OpenAI
# client = OpenAI()
# resp = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": "Giải thích GPT là gì trong 2 câu."}],
# )
# print(resp.choices[0].message.content)

d:\NLP\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dai\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 424.15it/s]


Artificial intelligence will change the world because we will be able to do it faster."
